# DIRE Kaggle — audit cây thư mục chỉ đọc

Notebook này chỉ kiểm kê cấu trúc của Dataset đã mount: đếm file/kích thước/suffix, tóm tắt theo split/domain, và liệt kê các cặp thư mục lồng cùng tên như `real/real` hoặc `adm/adm`.

Không đọc nội dung ảnh, không giải nén, không hash file, không sửa `/kaggle/input`, không tạo bản sao dữ liệu, không khởi tạo GPU. Report JSON duy nhất được ghi vào `/kaggle/working/dire-tree-audit-report.json`. Các cặp cùng tên là **ứng viên cần xem xét**, không tự động kết luận lỗi; các tầng split/domain/generator phải được giữ nguyên.

Chạy cell định nghĩa hàm rồi cell audit. Nếu auto-discovery báo nhiều/không có root, sửa `DIRE_ROOT` thủ công thành thư mục có `train/`, `val/`, `test/`. Dừng sau report; notebook này không pack/train.


In [ ]:
from __future__ import annotations

import os
from collections import Counter
from pathlib import Path
from typing import Any

SPLITS = {"train", "val", "test"}
IMAGE_SUFFIXES = {
    ".bmp", ".gif", ".jpeg", ".jpg", ".npy", ".npz", ".png",
    ".tif", ".tiff", ".webp",
}


def _new_stats() -> dict[str, Any]:
    return {"file_count": 0, "image_like_file_count": 0, "bytes": 0, "suffix_counts": Counter()}


def _suffix(name: str) -> str:
    lowered = name.lower()
    if lowered.endswith(".tar.gz"):
        return ".tar.gz"
    return Path(lowered).suffix or "<none>"


def audit_tree(root: str | Path) -> dict[str, Any]:
    """Count a mounted DIRE directory tree without opening or changing files."""
    root_path = Path(root).expanduser().resolve()
    if not root_path.is_dir():
        raise NotADirectoryError(f"DIRE root is not a directory: {root_path}")

    stats: dict[Path, dict[str, Any]] = {root_path: _new_stats()}
    directories = [root_path]
    pending = [root_path]
    repeated_pairs: list[tuple[Path, Path]] = []
    symlink_count = 0
    skipped_special_count = 0
    errors: list[dict[str, str]] = []

    while pending:
        current = pending.pop()
        stats.setdefault(current, _new_stats())
        try:
            with os.scandir(current) as entries:
                for entry in entries:
                    path = Path(entry.path)
                    try:
                        if entry.is_symlink():
                            symlink_count += 1
                        elif entry.is_dir(follow_symlinks=False):
                            stats.setdefault(path, _new_stats())
                            directories.append(path)
                            pending.append(path)
                            if path.name == current.name:
                                repeated_pairs.append((current, path))
                        elif entry.is_file(follow_symlinks=False):
                            size = entry.stat(follow_symlinks=False).st_size
                            row = stats[current]
                            suffix = _suffix(entry.name)
                            row["file_count"] += 1
                            row["bytes"] += size
                            row["suffix_counts"][suffix] += 1
                            row["image_like_file_count"] += suffix in IMAGE_SUFFIXES
                        else:
                            skipped_special_count += 1
                    except OSError as exc:
                        errors.append({"path": str(path), "error": str(exc)})
        except OSError as exc:
            errors.append({"path": str(current), "error": str(exc)})

    # Roll up directory totals bottom-up; counts are metadata-only, not image decoding.
    for directory in sorted(
        directories,
        key=lambda item: len(item.relative_to(root_path).parts),
        reverse=True,
    ):
        if directory == root_path:
            continue
        parent = directory.parent
        child_stats = stats[directory]
        parent_stats = stats[parent]
        parent_stats["file_count"] += child_stats["file_count"]
        parent_stats["image_like_file_count"] += child_stats["image_like_file_count"]
        parent_stats["bytes"] += child_stats["bytes"]
        parent_stats["suffix_counts"].update(child_stats["suffix_counts"])

    def relative(path: Path) -> str:
        return path.relative_to(root_path).as_posix() or "."

    same_name_nesting = []
    for parent, child in sorted(repeated_pairs, key=lambda pair: relative(pair[1])):
        child_stats = stats[child]
        same_name_nesting.append({
            "parent_directory": relative(parent),
            "child_directory": relative(child),
            "repeated_name": child.name,
            "file_count_in_child_subtree": child_stats["file_count"],
            "image_like_file_count_in_child_subtree": child_stats["image_like_file_count"],
            "bytes_in_child_subtree": child_stats["bytes"],
            "suffix_counts_in_child_subtree": dict(sorted(child_stats["suffix_counts"].items())),
        })

    split_domain_counts = []
    for directory in directories:
        parts = directory.relative_to(root_path).parts
        if len(parts) == 2 and parts[0] in SPLITS:
            row = stats[directory]
            split_domain_counts.append({
                "split": parts[0],
                "domain": parts[1],
                "file_count": row["file_count"],
                "image_like_file_count": row["image_like_file_count"],
                "bytes": row["bytes"],
                "suffix_counts": dict(sorted(row["suffix_counts"].items())),
            })

    total = stats[root_path]
    return {
        "schema_version": 1,
        "audit": "read-only DIRE directory-tree structure audit",
        "status": "complete" if not errors else "incomplete",
        "root": str(root_path),
        "input_modified": False,
        "file_contents_read": False,
        "gpu_work_started": False,
        "directory_count_including_root": len(directories),
        "regular_file_count": total["file_count"],
        "image_like_file_count_by_suffix_only": total["image_like_file_count"],
        "total_file_bytes_by_stat": total["bytes"],
        "suffix_counts": dict(sorted(total["suffix_counts"].items())),
        "split_domain_counts": sorted(split_domain_counts, key=lambda row: (row["split"], row["domain"])),
        "same_name_nesting_candidates": same_name_nesting,
        "symlinks_not_followed": symlink_count,
        "special_entries_skipped": skipped_special_count,
        "scan_errors": errors,
        "notes": [
            "Repeated directory names are reported as candidates, not automatically classified as errors.",
            "This audit counts regular files and suffixes; it does not decode images, hash files, or verify provenance.",
            "Symlinks are not followed. File byte totals are filesystem stat sizes.",
        ],
    }


In [ ]:
# Set this manually if auto-discovery finds zero or multiple matching roots.
from pathlib import Path
import json

input_root = Path('/kaggle/input')
mounted_roots = sorted(p for p in input_root.iterdir() if p.is_dir())
print('Mounted Kaggle inputs:')
for path in mounted_roots:
    print(' -', path)

candidates = []
for mounted in mounted_roots:
    for candidate in (mounted, mounted / 'dire'):
        if all((candidate / split).is_dir() for split in ('train', 'val', 'test')):
            resolved = candidate.resolve()
            if resolved not in candidates:
                candidates.append(resolved)
if len(candidates) != 1:
    raise RuntimeError(
        'Could not identify one DIRE root automatically. Set DIRE_ROOT to the folder '
        f'whose direct children include train/val/test. Candidates: {candidates}'
    )
DIRE_ROOT = candidates[0]

report = audit_tree(DIRE_ROOT)
REPORT_PATH = Path('/kaggle/working/dire-tree-audit-report.json')
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Audit status:', report['status'])
print('DIRE root:', report['root'])
print('Regular files:', report['regular_file_count'])
print('Image-like suffix count:', report['image_like_file_count_by_suffix_only'])
print('Same-name nested directories:', len(report['same_name_nesting_candidates']))
for row in report['same_name_nesting_candidates']:
    print(f" - {row['parent_directory']} -> {row['child_directory']}: "
          f"{row['file_count_in_child_subtree']} files, {row['bytes_in_child_subtree']} bytes")
print('Report written outside input:', REPORT_PATH)
